# Validation comparison: base vs GRPO (seed 1)

Compares the original model and the GRPO-trained model on the **500 validation questions** in `data/medqa_split_seed42.json`, to see whether GRPO changed the model at all (accuracy, calibration) before deciding whether seed 2 should use the same settings.

Eval modes (both models load in **bf16, no quantization**, same as GRPO training):

| Mode | Script | How | confidence |
|---|---|---|---|
| direct answer `direct` | `eval/evaluate.py --quantization none` | asks for just the letter, looks at the first token | probability of A-D |
| reason then answer `cot` | `eval/cot_eval.py` | **same prompt as GRPO training**, greedy, ends with `Answer: X` | probability of the letter token after `Answer:` |
| sampled vote `cot_vote8` | same | same prompt sampled 8 times (temperature 1.0, as in training) | share of the 8 samples that picked this answer |

`sample_acc` for `cot_vote8` (average accuracy of the sampled answers) can be compared directly with the training reward (about 0.50).

Run in order: check GPU -> set variables -> pull the repo -> install dependencies -> **restart session** -> rerun from "set variables" to the end. Use an **A100**; it takes about 20-30 minutes total.

Results go to `RESULTS_DIR` on Drive; finished evals get skipped automatically, so after a disconnect just rerun from the top.

In [ ]:
# 1. Check the GPU and Python version
!nvidia-smi
import sys, torch
print('Python', sys.version.split()[0])
print('torch', torch.__version__, '| CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    major, minor = torch.cuda.get_device_capability(0)
    print('GPU:', torch.cuda.get_device_name(0), f'| compute {major}.{minor} →', 'bf16' if major >= 8 else 'fp16')
assert torch.cuda.is_available(), 'No GPU: Runtime -> Change runtime type -> GPU'
assert sys.version_info >= (3, 11), 'TRL 1.16 needs Python >= 3.11'

In [ ]:
# 2. Set variables
REPO_URL = 'https://github.com/skjdsk/medqa-calibration.git'
BRANCH   = 'main'
REPO_DIR = '/content/medqa-calibration'
TRL_VERSION = '1.16.0'
USE_VLLM = True  # cell 4 installs the same vLLM version as training

DRIVE_ROOT = '/content/drive/MyDrive/medqa-calibration'
# adapters to evaluate: {model name: adapter folder}; base doesn't need one
ADAPTERS = {
    'grpo_seed1': f'{DRIVE_ROOT}/grpo/grpo_train5000_g8_ep1_len1024_seed1_bf16/adapter',
}
RESULTS_DIR = f'{DRIVE_ROOT}/eval_val'
N_SAMPLES   = 8      # samples for cot_vote (matches the 8 answers per question in training)
N_DEBUG     = None   # set to 20 to try a few questions first; use None (all 500) for the real comparison

In [ ]:
# 3. git clone / pull repo
# Private repo: add GITHUB_TOKEN (a GitHub personal access token) under the key icon (Secrets) on the left in Colab, and turn on "Notebook access"
import os, subprocess
from pathlib import Path

def sh(cmd, shown=None):
    print('$', shown or cmd)
    subprocess.run(cmd, shell=True, check=True)

clone_url = REPO_URL
try:
    from google.colab import userdata
    gh_token = userdata.get('GITHUB_TOKEN')
    if gh_token and REPO_URL.startswith('https://github.com/'):
        clone_url = REPO_URL.replace('https://', f'https://{gh_token}@')
except Exception:
    pass  # no GITHUB_TOKEN set, treat it as a public repo

if (Path(REPO_DIR) / '.git').exists():
    sh(f'git -C {REPO_DIR} fetch origin && git -C {REPO_DIR} checkout {BRANCH} && git -C {REPO_DIR} pull origin {BRANCH}')
else:
    sh(f'git clone --branch {BRANCH} {clone_url} {REPO_DIR}', shown=f'git clone --branch {BRANCH} {REPO_URL} {REPO_DIR}')
os.chdir(REPO_DIR)
sh('git log -1 --oneline')
for f in ['eval/evaluate.py', 'eval/cot_eval.py', 'eval/metrics.py', 'train/grpo/grpo_data.py', 'train/grpo/rewards.py', 'data/medqa_split_seed42.json']:
    assert Path(f).exists(), f'{f} not found in the repo: make sure it has been pushed to {BRANCH}'
print('cwd =', os.getcwd())

In [ ]:
# 4. Install dependencies
from pathlib import Path
ip = get_ipython()
if Path('requirements.txt').exists():
    ip.run_line_magic('pip', 'install -q -r requirements.txt')
ip.run_line_magic('pip', f'install -q "trl[peft,quantization]=={TRL_VERSION}" "matplotlib" "pandas" "pytest"')
if USE_VLLM:
    # vLLM installs the torch version it wants, so you usually need Runtime -> Restart session afterwards, then rerun from the "set variables" cell
    # vLLM 0.28+ dropped bitsandbytes quantization, but with QLoRA (4-bit) TRL has vLLM load the model through bitsandbytes
    # -> pinned to 0.27.1 (the last version with bitsandbytes, inside TRL 1.16's allowed 0.22-0.31 range, needs torch==2.13.0)
    ip.run_line_magic('pip', f'install -q "trl[vllm]=={TRL_VERSION}" "vllm==0.27.1"')

# Colab's preinstalled torchao 0.10 is too old: peft 0.21 errors out when adding LoRA without quantization (--quantization none)
# "only versions above 0.16.0 are supported". We don't use torchao (neither vllm nor peft needs it), so just uninstall it
ip.system('pip uninstall -y -q torchao')

import importlib.metadata as md
for pkg in ['torch', 'transformers', 'trl', 'peft', 'bitsandbytes', 'accelerate', 'datasets'] + (['vllm'] if USE_VLLM else []):
    try:
        print(f'{pkg:14s}', md.version(pkg))
    except md.PackageNotFoundError:
        print(f'{pkg:14s} not installed')
print('\nIf pip says a restart is needed above (or imports fail later): Runtime -> Restart session, then rerun from the "set variables" cell.')

**After installing: Runtime -> Restart session**, then run from cell 2 (set variables) down (skip cell 4).

In [ ]:
# 5. Mount Google Drive and check the adapters exist
import os
from pathlib import Path
from google.colab import drive
drive.mount('/content/drive')
os.chdir(REPO_DIR)
for name, path in ADAPTERS.items():
    ok = Path(path, 'adapter_config.json').exists()
    print(f'{name:12s}', '✅' if ok else '❌ not found', path)
    assert ok, f'adapter for {name} not found: check the path, or wait for Drive to finish syncing'
os.makedirs(RESULTS_DIR, exist_ok=True)
print('results folder:', RESULTS_DIR)

In [ ]:
# 6. (Optional) set HF_TOKEN: Qwen2.5 and MedQA are public so downloads work without it, but with it they're more reliable and less likely to get rate-limited
try:
    from google.colab import userdata
    token = userdata.get('HF_TOKEN')
    if token:
        os.environ['HF_TOKEN'] = token
        print('HF_TOKEN set')
except Exception:
    print('No HF_TOKEN found (optional), downloading anonymously')

In [ ]:
# 7. Unit tests (a few seconds)
!python -m pytest -q eval/test_cot_eval.py eval/test_metrics.py train/grpo/test_rewards.py

In [ ]:
# 8. Helper for running evals: skips anything whose result file already exists (so reruns after a disconnect don't redo work)
import subprocess
N_ARG = f' --n {N_DEBUG}' if N_DEBUG else ''

def run_eval(cmd, expected_csv):
    if Path(RESULTS_DIR, expected_csv).exists():
        print('already there, skipping:', expected_csv)
        return
    print('$', cmd)
    get_ipython().system(f'{cmd} 2>&1 | tee -a {RESULTS_DIR}/eval_log.txt')
    assert Path(RESULTS_DIR, expected_csv).exists(), f'{expected_csv} was not created: check the error above'

MODELS = {'base': None, **ADAPTERS}

In [ ]:
# 9. Direct answer (letter-logit), about 1-2 minutes per model
for name, adapter in MODELS.items():
    ad = f' --adapter {adapter}' if adapter else ''
    run_eval(f'python -m eval.evaluate --dataset medqa --medqa_subset val --quantization none'
             f'{ad} --model_name {name} --out_dir {RESULTS_DIR}{N_ARG}',
             f'preds_{name}_medqa4_val_direct.csv')

In [ ]:
# 10. Reason then answer (GRPO training prompt, vLLM), about 5-10 minutes per model
#     one process per model: vLLM only fully frees GPU memory when the process exits
for name, adapter in MODELS.items():
    ad = f' --adapter {adapter}' if adapter else ''
    run_eval(f'python -m eval.cot_eval{ad} --model_name {name} --n_samples {N_SAMPLES} '
             f'--out_dir {RESULTS_DIR}{N_ARG}',
             f'preds_{name}_medqa4_val_cot_vote{N_SAMPLES}.csv' if N_SAMPLES else f'preds_{name}_medqa4_val_cot.csv')

In [ ]:
# 11. Summary: accuracy, ECE, Brier, NLL, etc. (eval/metrics.py), plus a paired comparison
import pandas as pd
from IPython.display import display, Image

!python -m eval.metrics --pred_csv "{RESULTS_DIR}/preds_*_medqa4_val_*.csv" --out_dir {RESULTS_DIR}/metrics > /dev/null
s = pd.read_csv(f'{RESULTS_DIR}/metrics/summary.csv')
cols = ['eval_mode', 'model_name', 'n', 'accuracy', 'mean_confidence', 'ece', 'brier', 'nll',
        'hc_wrong_rate', 'mean_conf_wrong', 'mean_letter_mass']
display(s[cols].sort_values(['eval_mode', 'model_name']).round(4))

# extra info for cot: format, truncation, length; sample accuracy for cot_vote (~ training reward)
for mode, extra in [('cot', ['format_ok', 'truncated', 'n_completion_tokens']), (f'cot_vote{N_SAMPLES}', ['sample_acc', 'format_ok'])]:
    parts = [pd.read_csv(f'{RESULTS_DIR}/preds_{m}_medqa4_val_{mode}.csv') for m in MODELS
             if Path(RESULTS_DIR, f'preds_{m}_medqa4_val_{mode}.csv').exists()]
    if parts:
        print(mode); display(pd.concat(parts).groupby('model_name')[extra].mean().round(4))

# paired comparison: on the same questions, count base right / GRPO wrong vs base wrong / GRPO right
# if the two are close -> accuracy didn't really change; it only counts as a real change if |difference| > about 2 x sqrt(sum of both) (rough sign-test rule)
for mode in ['direct', 'cot', f'cot_vote{N_SAMPLES}']:
    p = Path(RESULTS_DIR, f'preds_base_medqa4_val_{mode}.csv')
    if not p.exists():
        continue
    b = pd.read_csv(p).set_index('question_id')
    for name in ADAPTERS:
        q = Path(RESULTS_DIR, f'preds_{name}_medqa4_val_{mode}.csv')
        if not q.exists():
            continue
        g = pd.read_csv(q).set_index('question_id').loc[b.index]
        lost, gained = int(((b.correct == 1) & (g.correct == 0)).sum()), int(((b.correct == 0) & (g.correct == 1)).sum())
        same = (b.pred == g.pred).mean()
        print(f'{mode:10s} {name}: same answer as base {same:.1%} | base right -> {name} wrong {lost} | base wrong -> {name} right {gained}'
              f' | net {gained - lost:+d} (rough significance line ±{2 * (lost + gained) ** 0.5:.0f})')

for f in sorted(Path(RESULTS_DIR, 'metrics').glob('reliability_*.png')):
    print(f.name); display(Image(str(f), width=420))

**How to read the results**

- **Answers mostly match base (say > 90%) and accuracy, ECE and Brier are all about the same** -> GRPO barely changed the model. Running seed 2 with the same settings isn't worth it; raise the learning rate first (e.g. 5e-5) and rerun both seeds with the new setting.
- **GRPO's accuracy / calibration clearly differs under cot or cot_vote** (net change beyond the significance line, or ECE differs by 0.02+) -> GRPO is doing something; go ahead and run seed 2 with the current settings.
- It's normal for `direct` and `cot` to disagree: GRPO was only trained on the CoT format.